In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import unicodedata

from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier

# =========================
# LOAD DATASETS
# =========================

match_df = pd.read_csv("champions_league_matches.csv")
player_df = pd.read_csv("attacking.csv")

# =========================
# CLEAN DATA
# =========================

match_df = match_df.dropna()

# =========================
# TRAIN MATCH MODEL
# =========================

encoder = LabelEncoder()

all_teams = pd.concat([
    match_df["home_team"],
    match_df["away_team"]
])

encoder.fit(all_teams)

match_df["home_team_encoded"] = encoder.transform(match_df["home_team"])
match_df["away_team_encoded"] = encoder.transform(match_df["away_team"])

X = match_df[[
    "home_team_encoded",
    "away_team_encoded"
]]

y = match_df["result"]

model = DecisionTreeClassifier(random_state=42)
model.fit(X, y)

print("⚽ Football AI Ready (Upgraded Version)")

# =========================
# SMART TEXT CLEANER
# =========================

def clean_text(text):
    return ''.join(
        c for c in unicodedata.normalize('NFKD', str(text))
        if not unicodedata.combining(c)
    ).lower()

# =========================
# MATCH PREDICTION (FIXED LOGIC)
# =========================

def predict_match(home_name, away_name):

    try:
        home_encoded = encoder.transform([home_name])[0]
        away_encoded = encoder.transform([away_name])[0]

        new_match = pd.DataFrame({
            "home_team_encoded": [home_encoded],
            "away_team_encoded": [away_encoded]
        })

        prediction = model.predict(new_match)[0]

        print("\n🏆 Result:", prediction)

    except:
        print("❌ Team not found in dataset")

# =========================
# PLAYER SEARCH (IMPROVED)
# =========================

def search_player(name):

    result = player_df[
        player_df["player_name"]
        .apply(clean_text)
        .str.contains(clean_text(name), na=False)
    ]

    if result.empty:
        print("❌ Player not found")
    else:
        print(result.head())

# =========================
# TOP ASSISTS
# =========================

def show_assists():

    top_players = player_df.sort_values(
        by="assists",
        ascending=False
    ).head(10)

    print(top_players[["player_name", "assists"]])

    top_players.plot(
        x="player_name",
        y="assists",
        kind="bar",
        figsize=(10,5)
    )

    plt.title("Top Assist Players")
    plt.show()

# =========================
# MAIN LOOP (SAME OLD STYLE)
# =========================

while True:

    query = input("\nType command (predict / player / assists / exit): ")

    if query.lower() == "exit":
        print("👋 Goodbye!")
        break

    elif query.lower().startswith("predict"):

        text = query.replace("predict", "").strip()

        try:
            home, away = text.split(" vs ")
            predict_match(home.strip(), away.strip())
        except:
            print("❌ Use format: predict TeamA vs TeamB")

    elif query.lower().startswith("player"):

        name = query.replace("player", "").strip()
        search_player(name)

    elif query.lower() == "assists":
        show_assists()

    else:
        print("Unknown command")